In [1]:

import os
import cv2

# Dataset path
dataset_path = "448"

crack_path = os.path.join(dataset_path, "Cracks")
noncrack_path = os.path.join(dataset_path, "NonCracks")

# Get image files
crack_images = [
    f for f in os.listdir(crack_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

noncrack_images = [
    f for f in os.listdir(noncrack_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Crack images:", len(crack_images))
print("Non-crack images:", len(noncrack_images))
print("Total images:", len(crack_images) + len(noncrack_images))

Crack images: 200
Non-crack images: 200
Total images: 400


In [2]:
import os
import cv2

# Select one image from each class
crack_file = os.path.join(crack_path, crack_images[0])
noncrack_file = os.path.join(noncrack_path, noncrack_images[0])

# Read images
crack_img = cv2.imread(crack_file)
noncrack_img = cv2.imread(noncrack_file)

# Display information
print("Crack Image")
print("-----------")
print("Shape:", crack_img.shape)
print("Data Type:", crack_img.dtype)
print("Channels:", crack_img.shape[2] if len(crack_img.shape) == 3 else 1)

print("\nNon-Crack Image")
print("---------------")
print("Shape:", noncrack_img.shape)
print("Data Type:", noncrack_img.dtype)
print("Channels:", noncrack_img.shape[2] if len(noncrack_img.shape) == 3 else 1)


Crack Image
-----------
Shape: (448, 448, 3)
Data Type: uint8
Channels: 3

Non-Crack Image
---------------
Shape: (448, 448, 3)
Data Type: uint8
Channels: 3


Resize and Convert All Images to Grayscale

In [3]:
import cv2

IMG_SIZE = (448, 448)

gray_images = []
labels = []
image_names = []

# Process Crack images
for filename in crack_images:
    path = os.path.join(crack_path, filename)

    img = cv2.imread(path)

    if img is not None:
        img = cv2.resize(img, IMG_SIZE)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        gray_images.append(gray)
        labels.append(1)
        image_names.append(filename)

# Process Non-Crack images
for filename in noncrack_images:
    path = os.path.join(noncrack_path, filename)

    img = cv2.imread(path)

    if img is not None:
        img = cv2.resize(img, IMG_SIZE)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        gray_images.append(gray)
        labels.append(0)
        image_names.append(filename)

print("Total processed images:", len(gray_images))
print("Image shape:", gray_images[0].shape)
print("Total labels:", len(labels))

Total processed images: 400
Image shape: (448, 448)
Total labels: 400


Apply Gaussian Blur

In [4]:
blurred_images = []

for gray in gray_images:
    blurred = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    blurred_images.append(blurred)

print("Total blurred images:", len(blurred_images))
print("Image shape:", blurred_images[0].shape)
print("Data type:", blurred_images[0].dtype)

Total blurred images: 400
Image shape: (448, 448)
Data type: uint8


Black-hat Morphological Transformation

In [5]:
import numpy as np

blackhat_images = []

# Create morphological kernel
kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (15, 15)
)

for blurred in blurred_images:
    blackhat = cv2.morphologyEx(
        blurred,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    blackhat_images.append(blackhat)

print("Total black-hat images:", len(blackhat_images))
print("Image shape:", blackhat_images[0].shape)
print("Data type:", blackhat_images[0].dtype)
print("Minimum value:", blackhat_images[0].min())
print("Maximum value:", blackhat_images[0].max())

Total black-hat images: 400
Image shape: (448, 448)
Data type: uint8
Minimum value: 0
Maximum value: 138


Normalize the Black-hat Images

In [6]:
normalized_blackhat_images = []

for blackhat in blackhat_images:
    normalized = blackhat.astype(np.float32) / 255.0
    normalized_blackhat_images.append(normalized)

print("Total normalized images:", len(normalized_blackhat_images))
print("Image shape:", normalized_blackhat_images[0].shape)
print("Data type:", normalized_blackhat_images[0].dtype)
print("Minimum value:", normalized_blackhat_images[0].min())
print("Maximum value:", normalized_blackhat_images[0].max())

Total normalized images: 400
Image shape: (448, 448)
Data type: float32
Minimum value: 0.0
Maximum value: 0.5411765


In [7]:
thresholded_images = []

THRESHOLD = 0.20

for normalized in normalized_blackhat_images:
    binary = (normalized >= THRESHOLD).astype(np.uint8)
    thresholded_images.append(binary)

print("Total thresholded images:", len(thresholded_images))
print("Image shape:", thresholded_images[0].shape)
print("Unique values:", np.unique(thresholded_images[0]))

Total thresholded images: 400
Image shape: (448, 448)
Unique values: [0 1]


In [8]:
import numpy as np
import pandas as pd

feature_rows = []

for i in range(len(gray_images)):

    # Original grayscale image
    gray = gray_images[i]

    # Black-hat representation
    blackhat = normalized_blackhat_images[i]

    # Thresholded crack-like regions
    binary = thresholded_images[i]

    # Canny edges
    edges = cv2.Canny(gray, 100, 200)

    # ---- Intensity features ----
    mean_intensity = np.mean(gray)
    std_intensity = np.std(gray)
    min_intensity = np.min(gray)
    max_intensity = np.max(gray)
    median_intensity = np.median(gray)

    # Pixel ratios
    dark_pixel_ratio = np.mean(gray < 50)
    bright_pixel_ratio = np.mean(gray > 200)

    # ---- Black-hat features ----
    blackhat_mean = np.mean(blackhat)
    blackhat_std = np.std(blackhat)
    blackhat_max = np.max(blackhat)

    # ---- Threshold feature ----
    threshold_pixel_ratio = np.mean(binary == 1)

    # ---- Edge feature ----
    edge_density = np.mean(edges > 0)

    # ---- Intensity percentiles ----
    p10 = np.percentile(gray, 10)
    p25 = np.percentile(gray, 25)
    p75 = np.percentile(gray, 75)
    p90 = np.percentile(gray, 90)

    feature_rows.append([
        mean_intensity,
        std_intensity,
        min_intensity,
        max_intensity,
        median_intensity,
        dark_pixel_ratio,
        bright_pixel_ratio,
        blackhat_mean,
        blackhat_std,
        blackhat_max,
        threshold_pixel_ratio,
        edge_density,
        p10,
        p25,
        p75,
        p90,
        labels[i],
        image_names[i]
    ])

# Create DataFrame
features_df = pd.DataFrame(
    feature_rows,
    columns=[
        "mean_intensity",
        "std_intensity",
        "min_intensity",
        "max_intensity",
        "median_intensity",
        "dark_pixel_ratio",
        "bright_pixel_ratio",
        "blackhat_mean",
        "blackhat_std",
        "blackhat_max",
        "threshold_pixel_ratio",
        "edge_density",
        "p10",
        "p25",
        "p75",
        "p90",
        "label",
        "image_name"
    ]
)

print("Feature table shape:", features_df.shape)
features_df.head()

Feature table shape: (400, 18)


,mean_intensity,std_intensity,min_intensity,max_intensity,median_intensity,dark_pixel_ratio,bright_pixel_ratio,blackhat_mean,blackhat_std,blackhat_max,threshold_pixel_ratio,edge_density,p10,p25,p75,p90,label,image_name
0,122.096147,38.014345,0,255,121.0,0.016333,0.026676,0.111194,0.078720,0.541176,0.140406,0.361652,74.0,95.0,147.0,172.0,1,IMG_20180513_164959951_HDR resized_448.jpg
1,131.822704,29.319148,0,255,133.0,0.006741,0.011759,0.079945,0.060100,0.454902,0.047234,0.327711,95.0,114.0,150.0,167.0,1,IMG_20180513_165129852_HDR resized_448.jpg
2,121.171217,35.572563,0,255,120.0,0.024992,0.020712,0.096169,0.083800,0.639216,0.119489,0.314827,77.0,100.0,141.0,168.0,1,IMG_20180513_165150613_HDR resized_448.jpg
3,130.117701,32.745086,1,255,131.0,0.008889,0.014917,0.097188,0.068227,0.533333,0.085140,0.357083,88.0,108.0,152.0,170.0,1,IMG_20180513_165345878_HDR resized_448.jpg
4,131.079998,33.199356,0,255,132.0,0.009741,0.016791,0.099384,0.070669,0.533333,0.095050,0.357337,88.0,109.0,153.0,172.0,1,IMG_20180513_165349788_HDR resized_448.jpg


In [9]:
print("Feature table shape:", features_df.shape)

print("\nColumn names:")
print(features_df.columns.tolist())

print("\nFirst 5 rows:")
display(features_df.head())

Feature table shape: (400, 18)

Column names:
['mean_intensity', 'std_intensity', 'min_intensity', 'max_intensity', 'median_intensity', 'dark_pixel_ratio', 'bright_pixel_ratio', 'blackhat_mean', 'blackhat_std', 'blackhat_max', 'threshold_pixel_ratio', 'edge_density', 'p10', 'p25', 'p75', 'p90', 'label', 'image_name']

First 5 rows:


,mean_intensity,std_intensity,min_intensity,max_intensity,median_intensity,dark_pixel_ratio,bright_pixel_ratio,blackhat_mean,blackhat_std,blackhat_max,threshold_pixel_ratio,edge_density,p10,p25,p75,p90,label,image_name
0,122.096147,38.014345,0,255,121.0,0.016333,0.026676,0.111194,0.078720,0.541176,0.140406,0.361652,74.0,95.0,147.0,172.0,1,IMG_20180513_164959951_HDR resized_448.jpg
1,131.822704,29.319148,0,255,133.0,0.006741,0.011759,0.079945,0.060100,0.454902,0.047234,0.327711,95.0,114.0,150.0,167.0,1,IMG_20180513_165129852_HDR resized_448.jpg
2,121.171217,35.572563,0,255,120.0,0.024992,0.020712,0.096169,0.083800,0.639216,0.119489,0.314827,77.0,100.0,141.0,168.0,1,IMG_20180513_165150613_HDR resized_448.jpg
3,130.117701,32.745086,1,255,131.0,0.008889,0.014917,0.097188,0.068227,0.533333,0.085140,0.357083,88.0,108.0,152.0,170.0,1,IMG_20180513_165345878_HDR resized_448.jpg
4,131.079998,33.199356,0,255,132.0,0.009741,0.016791,0.099384,0.070669,0.533333,0.095050,0.357337,88.0,109.0,153.0,172.0,1,IMG_20180513_165349788_HDR resized_448.jpg


In [10]:
print("Missing values:")
print(features_df.isnull().sum())

print("\nTotal missing values:", features_df.isnull().sum().sum())

Missing values:
mean_intensity           0
std_intensity            0
min_intensity            0
max_intensity            0
median_intensity         0
dark_pixel_ratio         0
bright_pixel_ratio       0
blackhat_mean            0
blackhat_std             0
blackhat_max             0
threshold_pixel_ratio    0
edge_density             0
p10                      0
p25                      0
p75                      0
p90                      0
label                    0
image_name               0
dtype: int64

Total missing values: 0


In [11]:
feature_columns = [
    "mean_intensity",
    "std_intensity",
    "min_intensity",
    "max_intensity",
    "median_intensity",
    "dark_pixel_ratio",
    "bright_pixel_ratio",
    "blackhat_mean",
    "blackhat_std",
    "blackhat_max",
    "threshold_pixel_ratio",
    "edge_density",
    "p10",
    "p25",
    "p75",
    "p90"
]

class_statistics = features_df.groupby("label")[feature_columns].mean()

print(class_statistics)

       mean_intensity  std_intensity  min_intensity  max_intensity  \
label                                                                
0          157.502980      28.756693         38.805         254.90   
1          144.244057      35.310512         15.590         254.91   

       median_intensity  dark_pixel_ratio  bright_pixel_ratio  blackhat_mean  \
label                                                                          
0               157.295          0.003182            0.073852       0.086788   
1               144.290          0.014283            0.094933       0.114120   

       blackhat_std  blackhat_max  threshold_pixel_ratio  edge_density  \
label                                                                    
0          0.053526      0.379412               0.069924      0.287241   
1          0.078961      0.558490               0.151971      0.338396   

           p10      p25      p75       p90  
label                                       
0      120.

In [12]:
features_df.to_csv(
    "asphalt_improved_features.csv",
    index=False
)

print("Feature dataset saved successfully.")
print("File: asphalt_improved_features.csv")

Feature dataset saved successfully.
File: asphalt_improved_features.csv


In [13]:
X = features_df[feature_columns]
y = features_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (400, 16)
y shape: (400,)


In [14]:
print("Feature columns:")
for i, col in enumerate(X.columns, start=1):
    print(i, col)

Feature columns:
1 mean_intensity
2 std_intensity
3 min_intensity
4 max_intensity
5 median_intensity
6 dark_pixel_ratio
7 bright_pixel_ratio
8 blackhat_mean
9 blackhat_std
10 blackhat_max
11 threshold_pixel_ratio
12 edge_density
13 p10
14 p25
15 p75
16 p90


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training features: (320, 16)
Testing features : (80, 16)

Training class distribution:
label
1    160
0    160
Name: count, dtype: int64

Testing class distribution:
label
0    40
1    40
Name: count, dtype: int64


Standardize the Features

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Testing shape :", X_test_scaled.shape)

print("\nTraining mean of first feature:",
      X_train_scaled[:, 0].mean())

print("Training std of first feature:",
      X_train_scaled[:, 0].std())

Training shape: (320, 16)
Testing shape : (80, 16)

Training mean of first feature: 3.885780586188048e-16
Training std of first feature: 1.0


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import time

# -------------------------------
# Logistic Regression
# -------------------------------
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

start_time = time.perf_counter()
lr_model.fit(X_train_scaled, y_train)
lr_train_time = time.perf_counter() - start_time

start_time = time.perf_counter()
lr_predictions = lr_model.predict(X_test_scaled)
lr_prediction_time = time.perf_counter() - start_time


# -------------------------------
# Random Forest
# -------------------------------
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start_time = time.perf_counter()
rf_model.fit(X_train, y_train)
rf_train_time = time.perf_counter() - start_time

start_time = time.perf_counter()
rf_predictions = rf_model.predict(X_test)
rf_prediction_time = time.perf_counter() - start_time


# -------------------------------
# Timing Results
# -------------------------------
print("Model Training and Prediction Completed\n")

print("Logistic Regression")
print("Training time :", round(lr_train_time, 6), "seconds")
print("Prediction time:", round(lr_prediction_time, 6), "seconds")

print("\nRandom Forest")
print("Training time :", round(rf_train_time, 6), "seconds")
print("Prediction time:", round(rf_prediction_time, 6), "seconds")

Model Training and Prediction Completed

Logistic Regression
Training time : 0.098572 seconds
Prediction time: 0.000504 seconds

Random Forest
Training time : 0.158942 seconds
Prediction time: 0.026518 seconds


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Logistic Regression
lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_precision = precision_score(y_test, lr_predictions)
lr_recall = recall_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions)
lr_cm = confusion_matrix(y_test, lr_predictions)

# Random Forest
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)
rf_cm = confusion_matrix(y_test, rf_predictions)


print("========== LOGISTIC REGRESSION ==========")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))
print("Confusion Matrix:")
print(lr_cm)

print("\n========== RANDOM FOREST ==========")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("Confusion Matrix:")
print(rf_cm)

========== LOGISTIC REGRESSION ==========
Accuracy : 0.925
Precision: 0.9474
Recall   : 0.9
F1 Score : 0.9231
Confusion Matrix:
[[38  2]
 [ 4 36]]

========== RANDOM FOREST ==========
Accuracy : 0.9625
Precision: 0.9512
Recall   : 0.975
F1 Score : 0.963
Confusion Matrix:
[[38  2]
 [ 1 39]]


In [19]:
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy
    ],
    "Precision": [
        lr_precision,
        rf_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall
    ],
    "F1 Score": [
        lr_f1,
        rf_f1
    ],
    "Training Time (sec)": [
        lr_train_time,
        rf_train_time
    ],
    "Prediction Time (sec)": [
        lr_prediction_time,
        rf_prediction_time
    ]
})

comparison_df = comparison_df.round(4)

display(comparison_df)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (sec),Prediction Time (sec)
0,Logistic Regression,0.9250,0.9474,0.900,0.9231,0.0986,0.0005
1,Random Forest,0.9625,0.9512,0.975,0.9630,0.1589,0.0265
